# Weekend 6 — SQL + Django ORM + migracje

## Plik: cwiczenia_clean.ipynb

To jest wersja ćwiczeniowa bez rozwiązań. Pracujesz na realnym datasecie e-commerce, wykonujesz zapytania SQL, projektujesz model relacyjny, a potem mapujesz go na Django models i podstawowe QuerySets.

Główna zasada: najpierw rozumiemy dane i relacje, dopiero potem piszemy kod.

## Opis datasetu

### Nazwa

Olist Brazilian E-Commerce Public Dataset

### Źródło

https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

### Kontekst biznesowy

Dataset opisuje marketplace e-commerce. Zawiera klientów, zamówienia, produkty, sprzedawców, płatności, opinie i pozycje zamówień. Dzięki temu można ćwiczyć typowe problemy backendowe: filtrowanie danych, relacje między tabelami, agregacje, raporty, projektowanie schematu oraz mapowanie tabel na Django models.

### Główne tabele i kolumny

#### customers

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| customer_id | text | identyfikator klienta w kontekście zamówienia |
| customer_unique_id | text | unikalny identyfikator klienta |
| customer_zip_code_prefix | integer | prefiks kodu pocztowego |
| customer_city | text | miasto klienta |
| customer_state | text | stan/region klienta |

#### orders

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| order_id | text | identyfikator zamówienia |
| customer_id | text | klient przypisany do zamówienia |
| order_status | text | status zamówienia |
| order_purchase_timestamp | datetime | data złożenia zamówienia |
| order_approved_at | datetime | data akceptacji płatności |
| order_delivered_carrier_date | datetime | data przekazania przewoźnikowi |
| order_delivered_customer_date | datetime | data dostawy do klienta |
| order_estimated_delivery_date | datetime | planowana data dostawy |

#### order_items

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| order_id | text | zamówienie |
| order_item_id | integer | numer pozycji w zamówieniu |
| product_id | text | produkt |
| seller_id | text | sprzedawca |
| shipping_limit_date | datetime | graniczna data wysyłki |
| price | decimal | cena produktu |
| freight_value | decimal | koszt dostawy |

#### products

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| product_id | text | identyfikator produktu |
| product_category_name | text | kategoria produktu |
| product_name_lenght | integer | długość nazwy produktu |
| product_description_lenght | integer | długość opisu produktu |
| product_photos_qty | integer | liczba zdjęć produktu |
| product_weight_g | integer | waga produktu w gramach |
| product_length_cm | integer | długość produktu |
| product_height_cm | integer | wysokość produktu |
| product_width_cm | integer | szerokość produktu |

#### sellers

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| seller_id | text | identyfikator sprzedawcy |
| seller_zip_code_prefix | integer | prefiks kodu pocztowego |
| seller_city | text | miasto sprzedawcy |
| seller_state | text | stan/region sprzedawcy |

#### order_payments

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| order_id | text | zamówienie |
| payment_sequential | integer | numer płatności |
| payment_type | text | typ płatności |
| payment_installments | integer | liczba rat |
| payment_value | decimal | wartość płatności |

#### order_reviews

| Kolumna | Typ | Opis biznesowy |
|---|---|---|
| review_id | text | identyfikator opinii |
| order_id | text | zamówienie |
| review_score | integer | ocena 1–5 |
| review_comment_title | text | tytuł opinii |
| review_comment_message | text | treść opinii |
| review_creation_date | datetime | data wystawienia opinii |
| review_answer_timestamp | datetime | data odpowiedzi |

### Etykiety i kategorie

- order_status: delivered, shipped, canceled, invoiced, processing, unavailable, approved, created
- payment_type: credit_card, boleto, voucher, debit_card, not_defined
- review_score: ocena od 1 do 5
- product_category_name: kategoria produktu
- customer_state i seller_state: regiony geograficzne

## Jak pracować z tym notebookiem

1. Najpierw uruchom sekcję setupu.
2. Upewnij się, że pliki CSV znajdują się w katalogu `data/`.
3. Wykonuj zadania po kolei.
4. Przy zadaniach SQL wpisuj zapytanie w zmiennej `query` i uruchamiaj `run_sql(query)`.
5. Przy zadaniach projektowych zapisuj swoje decyzje w komórkach markdown albo jako komentarze.
6. Przy zadaniach Django nie musisz uruchamiać kodu w tym notebooku, jeśli pracujesz bezpośrednio w projekcie Django. Notebook służy wtedy jako instrukcja i miejsce na notatki.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

DATA_DIR = Path("data")
DB_PATH = Path("olist_workshop.sqlite3")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Data dir:", DATA_DIR.resolve())
print("Database path:", DB_PATH.resolve())

## Przygotowanie danych

W katalogu `data/` powinny znajdować się pliki CSV z datasetu Olist. Jeżeli nazwy plików różnią się od poniższych, popraw mapowanie w słowniku `csv_files`.

To ćwiczenie używa SQLite, żeby nie tracić czasu na konfigurację. W projekcie końcowym możesz pracować na PostgreSQL.

In [ ]:
csv_files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
}

# TODO: Uruchom tę komórkę i sprawdź, czy wszystkie pliki istnieją.
for table_name, file_name in csv_files.items():
    path = DATA_DIR / file_name
    print(f"{table_name:15s} -> {path} | exists={path.exists()}")

In [ ]:
def load_csvs_to_sqlite(csv_mapping: dict[str, str], data_dir: Path, db_path: Path) -> sqlite3.Connection:
    """Ładuje pliki CSV do SQLite jako osobne tabele."""
    connection = sqlite3.connect(db_path)
    
    for table_name, file_name in csv_mapping.items():
        path = data_dir / file_name
        if not path.exists():
            print(f"Pominięto {table_name}: brak pliku {path}")
            continue
        
        # TODO: Wczytaj CSV do DataFrame.
        # df = ...
        
        # TODO: Zapisz DataFrame jako tabelę SQL.
        # df.to_sql(...)
        
        print(f"Załadowano tabelę: {table_name}")
    
    return connection


# TODO: Odkomentuj po uzupełnieniu funkcji.
# conn = load_csvs_to_sqlite(csv_files, DATA_DIR, DB_PATH)

In [ ]:
# Jeżeli prowadzący dostarczył gotową bazę SQLite, wystarczy połączyć się tak:
conn = sqlite3.connect(DB_PATH)

def run_sql(query: str, connection: sqlite3.Connection = conn) -> pd.DataFrame:
    """Uruchamia zapytanie SQL i zwraca wynik jako DataFrame."""
    return pd.read_sql_query(query, connection)

# TODO: Sprawdź listę tabel w bazie.
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

# run_sql(query)

# Część 1 — SQL: SELECT, WHERE, ORDER BY, LIMIT

## Zadanie 1 — Pierwszy SELECT z tabeli `orders`

- Biznes: support chce podejrzeć próbkę zamówień.
- Cel techniczny: `SELECT`, `FROM`, `LIMIT`.
- Trudność: 1/5
- Szacowany czas: 8 min

Wybierz 10 rekordów z tabeli `orders`. Nie używaj jeszcze wszystkich kolumn. Zwróć: `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp`.

Wskazówki:
- Zacznij od `SELECT`.
- Pamiętaj o `FROM orders`.
- Użyj `LIMIT 10`.

In [ ]:
query = """
-- TODO: wybierz 10 zamówień z wybranymi kolumnami

"""

# run_sql(query)

## Zadanie 2 — Filtrowanie zamówień po statusie

- Biznes: dział operacyjny analizuje tylko zamówienia dostarczone.
- Cel techniczny: `WHERE`.
- Trudność: 1/5
- Szacowany czas: 8 min

Pobierz zamówienia, dla których `order_status = 'delivered'`. Zwróć `order_id`, `customer_id`, `order_status`.

Wskazówki:
- Warunek wpisz po `WHERE`.
- Ogranicz wynik do 20 rekordów.
- Sprawdź, czy w wyniku faktycznie są tylko dostarczone zamówienia.

In [ ]:
query = """
-- TODO: pobierz dostarczone zamówienia

"""

# run_sql(query)

## Zadanie 3 — Sortowanie najnowszych zamówień

- Biznes: panel admina ma pokazywać ostatnie zakupy.
- Cel techniczny: `ORDER BY`, `DESC`, `LIMIT`.
- Trudność: 1/5
- Szacowany czas: 8 min

Pobierz 15 najnowszych zamówień według `order_purchase_timestamp`.

Wskazówki:
- Sortowanie malejące to `DESC`.
- Użyj aliasu tylko wtedy, gdy poprawia czytelność.
- Zwróć datę zakupu, żeby zweryfikować sortowanie.

In [ ]:
query = """
-- TODO: pobierz 15 najnowszych zamówień

"""

# run_sql(query)

## Zadanie 4 — Zamówienia z konkretnego okresu

- Biznes: zespół sprzedaży analizuje wybrany miesiąc.
- Cel techniczny: filtrowanie po dacie.
- Trudność: 2/5
- Szacowany czas: 10 min

Pobierz zamówienia złożone w styczniu 2018 roku. Zwróć `order_id`, `order_status`, `order_purchase_timestamp`.

Wskazówki:
- Możesz użyć `>= '2018-01-01'` i `< '2018-02-01'`.
- To często bezpieczniejsze niż `BETWEEN` przy datach z godziną.
- Posortuj wynik rosnąco po dacie zakupu.

In [ ]:
query = """
-- TODO: pobierz zamówienia ze stycznia 2018

"""

# run_sql(query)

## Zadanie 5 — Klienci z wybranego stanu

- Biznes: marketing planuje kampanię regionalną.
- Cel techniczny: filtrowanie tekstu i sortowanie.
- Trudność: 1/5
- Szacowany czas: 8 min

Pobierz klientów ze stanu `SP`. Zwróć `customer_id`, `customer_city`, `customer_state` i posortuj po mieście.

Wskazówki:
- Użyj tabeli `customers`.
- Warunek dotyczy kolumny `customer_state`.
- Ogranicz wynik do 30 rekordów.

In [ ]:
query = """
-- TODO: pobierz klientów ze stanu SP

"""

# run_sql(query)

# Część 2 — SQL: JOIN

## Zadanie 6 — JOIN `orders` z `customers`

- Biznes: support chce widzieć zamówienie razem z lokalizacją klienta.
- Cel techniczny: `INNER JOIN`.
- Trudność: 2/5
- Szacowany czas: 15 min

Połącz `orders` z `customers` po `customer_id`. Zwróć `order_id`, `order_status`, `customer_city`, `customer_state`.

Wskazówki:
- Użyj aliasów `o` i `c`.
- Warunek połączenia wpisz po `ON`.
- Ogranicz wynik do 20 rekordów.

In [ ]:
query = """
-- TODO: połącz orders z customers

"""

# run_sql(query)

## Zadanie 7 — JOIN `orders`, `order_items`, `products`

- Biznes: analityk chce sprawdzić, jakie kategorie produktów pojawiają się w zamówieniach.
- Cel techniczny: JOIN wielu tabel.
- Trudność: 3/5
- Szacowany czas: 18 min

Połącz `orders`, `order_items` i `products`. Zwróć `order_id`, `order_status`, `product_id`, `product_category_name`, `price`.

Wskazówki:
- Najpierw połącz `orders` z `order_items`.
- Potem dołącz `products`.
- Pamiętaj, że jedno zamówienie może mieć wiele pozycji.

In [ ]:
query = """
-- TODO: połącz orders, order_items i products

"""

# run_sql(query)

## Zadanie 8 — JOIN z płatnościami

- Biznes: finance chce analizować typy płatności.
- Cel techniczny: JOIN `orders` + `order_payments`.
- Trudność: 2/5
- Szacowany czas: 12 min

Połącz zamówienia z płatnościami. Zwróć `order_id`, `order_status`, `payment_type`, `payment_installments`, `payment_value`.

Wskazówki:
- Łącz po `order_id`.
- Jedno zamówienie może mieć więcej niż jedną płatność.
- Posortuj po `payment_value` malejąco.

In [ ]:
query = """
-- TODO: połącz orders z order_payments

"""

# run_sql(query)

## Zadanie 9 — LEFT JOIN dla zamówień bez opinii

- Biznes: zespół customer success chce znaleźć zamówienia bez recenzji.
- Cel techniczny: `LEFT JOIN`, `IS NULL`.
- Trudność: 3/5
- Szacowany czas: 18 min

Znajdź zamówienia, które nie mają opinii w tabeli `order_reviews`.

Wskazówki:
- Zacznij od tabeli `orders`.
- Dołącz `order_reviews` przez `LEFT JOIN`.
- Szukaj rekordów, gdzie `review_id IS NULL`.

In [ ]:
query = """
-- TODO: znajdź zamówienia bez opinii

"""

# run_sql(query)

## Zadanie 10 — Debugowanie błędnego JOIN-a

- Biznes: raport pokazuje podejrzanie dużo rekordów.
- Cel techniczny: diagnoza złego warunku JOIN.
- Trudność: 3/5
- Szacowany czas: 15 min

Poniżej znajduje się celowo podejrzany JOIN. Nie uruchamiaj go bez `LIMIT`. Twoim zadaniem jest wyjaśnić, co jest w nim nie tak, a potem napisać poprawną wersję.

Podejrzany pomysł: łączyć `orders.order_id` z `customers.customer_id`.

Wskazówki:
- Porównaj znaczenie biznesowe kolumn.
- Sprawdź właściwą relację między `orders` i `customers`.
- Najpierw testuj małe próbki.

In [ ]:
query = """
-- TODO: napisz poprawną wersję JOIN-a orders + customers

"""

# run_sql(query)

# Część 3 — SQL: GROUP BY i agregacje

## Zadanie 11 — Liczba zamówień po statusie

- Biznes: dashboard operacyjny pokazuje rozkład statusów.
- Cel techniczny: `GROUP BY`, `COUNT`.
- Trudność: 2/5
- Szacowany czas: 10 min

Policz liczbę zamówień dla każdego `order_status`.

Wskazówki:
- Grupuj po `order_status`.
- Nadaj alias, np. `orders_count`.
- Posortuj wynik malejąco po liczbie zamówień.

In [ ]:
query = """
-- TODO: policz zamówienia po statusie

"""

# run_sql(query)

## Zadanie 12 — Przychód po kategoriach produktów

- Biznes: dział sprzedaży chce poznać najlepsze kategorie.
- Cel techniczny: `SUM`, JOIN, `GROUP BY`.
- Trudność: 3/5
- Szacowany czas: 20 min

Policz sumę `price` dla każdej kategorii produktu. Pomiń rekordy bez kategorii.

Wskazówki:
- Użyj `order_items` i `products`.
- Łącz po `product_id`.
- Zastanów się, czy `freight_value` powinno być częścią przychodu. W tym zadaniu licz tylko `price`.

In [ ]:
query = """
-- TODO: policz przychód po kategoriach produktów

"""

# run_sql(query)

## Zadanie 13 — Średni koszt dostawy po stanie klienta

- Biznes: logistyka porównuje regiony.
- Cel techniczny: `AVG`, JOIN wielu tabel.
- Trudność: 3/5
- Szacowany czas: 18 min

Policz średni `freight_value` dla każdego `customer_state`.

Wskazówki:
- Potrzebujesz `orders`, `customers`, `order_items`.
- Grupuj po `customer_state`.
- Posortuj od najwyższego średniego kosztu dostawy.

In [ ]:
query = """
-- TODO: policz średni koszt dostawy po stanie klienta

"""

# run_sql(query)

## Zadanie 14 — Najlepsi sprzedawcy według przychodu

- Biznes: marketplace chce znaleźć top sellerów.
- Cel techniczny: agregacja po `seller_id`.
- Trudność: 3/5
- Szacowany czas: 15 min

Policz sumę `price` dla każdego sprzedawcy i pokaż top 10.

Wskazówki:
- Użyj `order_items`.
- Grupuj po `seller_id`.
- Użyj `LIMIT 10`.

In [ ]:
query = """
-- TODO: znajdź top 10 sprzedawców według przychodu

"""

# run_sql(query)

## Zadanie 15 — Kategorie ze średnią oceną poniżej 3

- Biznes: zespół jakości chce znaleźć problematyczne kategorie.
- Cel techniczny: JOIN, `AVG`, `HAVING`.
- Trudność: 4/5
- Szacowany czas: 25 min

Znajdź kategorie produktów, których średnia ocena `review_score` jest mniejsza niż 3.

Wskazówki:
- Potrzebujesz `products`, `order_items`, `orders`, `order_reviews`.
- `WHERE` filtruje rekordy przed grupowaniem.
- `HAVING` filtruje grupy po agregacji.

In [ ]:
query = """
-- TODO: znajdź kategorie ze średnią oceną poniżej 3

"""

# run_sql(query)

## Zadanie 16 — Raport miesięczny sprzedaży

- Biznes: zarząd potrzebuje raportu miesięcznego.
- Cel techniczny: grupowanie po miesiącu.
- Trudność: 4/5
- Szacowany czas: 25 min

Przygotuj raport miesięczny: miesiąc zakupu, liczba zamówień i suma wartości pozycji zamówień.

Wskazówki:
- W SQLite możesz użyć `strftime('%Y-%m', order_purchase_timestamp)`.
- Do wartości sprzedaży użyj `order_items.price`.
- Pamiętaj, że po dołączeniu `order_items` liczysz pozycje zamówień, nie same zamówienia. Do liczby zamówień może przydać się `COUNT(DISTINCT ...)`.

In [ ]:
query = """
-- TODO: przygotuj miesięczny raport sprzedaży

"""

# run_sql(query)

# Część 4 — Projektowanie modelu relacyjnego

## Zadanie 17 — Identyfikacja encji

- Biznes: projektujesz własny moduł zamówień do aplikacji webowej.
- Cel techniczny: rozpoznanie kandydatów na tabele.
- Trudność: 2/5
- Szacowany czas: 15 min

Na podstawie datasetu i przykładowej domeny e-commerce wypisz główne encje. Oddziel encje od zwykłych atrybutów.

Wskazówki:
- Szukaj rzeczowników: klient, zamówienie, produkt, płatność.
- Nie każdy atrybut musi być tabelą.
- Zastanów się, które obiekty będą miały własny cykl życia.

### Twoje notatki

- Encje:
  - TODO
- Atrybuty:
  - TODO
- Wątpliwości projektowe:
  - TODO

## Zadanie 18 — Relacje 1:N

- Biznes: klient może mieć wiele zamówień, a sprzedawca wiele pozycji sprzedażowych.
- Cel techniczny: wskazanie strony klucza obcego.
- Trudność: 2/5
- Szacowany czas: 12 min

Wypisz minimum 3 relacje 1:N z datasetu. Przy każdej napisz, po której stronie powinien znaleźć się `ForeignKey`.

Wskazówki:
- Pytaj: po której stronie jest wiele rekordów?
- FK zwykle trafia po stronie „wiele”.
- Używaj nazw tabel i kolumn.

### Twoje notatki

| Relacja | Strona 1 | Strona N | Gdzie FK? |
|---|---|---|---|
| TODO | TODO | TODO | TODO |
| TODO | TODO | TODO | TODO |
| TODO | TODO | TODO | TODO |

## Zadanie 19 — Relacja N:M i model pośredni

- Biznes: zamówienie ma wiele produktów, produkt może wystąpić w wielu zamówieniach.
- Cel techniczny: zaprojektowanie tabeli pośredniej.
- Trudność: 3/5
- Szacowany czas: 20 min

Wyjaśnij, dlaczego relacji `Order` ↔ `Product` nie zapisujemy jako listy produktów w jednej kolumnie. Zaprojektuj model pośredni `OrderItem`.

Wskazówki:
- Relacja ma własne dane: cena, ilość, sprzedawca.
- Tabela pośrednia może być pełnoprawną encją.
- Zastanów się, jaki powinien być klucz główny lub unikalność.

### Twoje notatki

Proponowane pola `OrderItem`:

| Pole | Typ | Po co jest? |
|---|---|---|
| TODO | TODO | TODO |
| TODO | TODO | TODO |
| TODO | TODO | TODO |

## Zadanie 20 — Szkic ERD dla aplikacji e-commerce

- Biznes: przygotowujesz schemat pod backend API.
- Cel techniczny: prosty diagram encji i relacji.
- Trudność: 3/5
- Szacowany czas: 25 min

Narysuj prosty ERD dla minimum 4 encji: `Customer`, `Order`, `Product`, `OrderItem`. Możesz użyć kartki, Miro, draw.io albo opisu tekstowego.

Wskazówki:
- Zaznacz PK i FK.
- Zaznacz relacje 1:N.
- Nie komplikuj modelu za wcześnie.

### Opis tekstowy ERD

- `Customer`:
  - PK: TODO
  - pola: TODO
- `Order`:
  - PK: TODO
  - FK: TODO
  - pola: TODO
- `Product`:
  - PK: TODO
  - pola: TODO
- `OrderItem`:
  - PK: TODO
  - FK: TODO
  - pola: TODO

## Zadanie 21 — Mapowanie tabel na Django models

- Biznes: przenosisz schemat do aplikacji Django.
- Cel techniczny: tabela → model, kolumna → field.
- Trudność: 2/5
- Szacowany czas: 15 min

Dla encji `Customer`, `Product`, `Order`, `OrderItem` zapisz proponowane klasy Django i typy pól. Na razie nie musisz pisać pełnego kodu.

Wskazówki:
- Nazwy modeli pisz w liczbie pojedynczej.
- Pole relacji nazwij `customer`, nie `customer_id`.
- Django samo doda kolumnę `customer_id` w bazie.

### Twoje mapowanie

| Model | Pole | Typ Django | Uwagi |
|---|---|---|---|
| Customer | TODO | TODO | TODO |
| Product | TODO | TODO | TODO |
| Order | TODO | TODO | TODO |
| OrderItem | TODO | TODO | TODO |

## Zadanie 22 — `ManyToManyField` czy model pośredni?

- Biznes: pozycja zamówienia ma cenę i ilość.
- Cel techniczny: decyzja projektowa.
- Trudność: 3/5
- Szacowany czas: 15 min

Zdecyduj, czy w tym przypadku użyć prostego `ManyToManyField`, czy osobnego modelu `OrderItem`. Uzasadnij decyzję.

Wskazówki:
- Jeśli relacja ma własne dane, zwykle potrzebujesz modelu pośredniego.
- `quantity` i `unit_price` należą do relacji zamówienie-produkt.
- W realnych sklepach cena produktu w zamówieniu musi zostać zapamiętana historycznie.

### Decyzja

- Wybrane rozwiązanie: TODO
- Uzasadnienie biznesowe: TODO
- Uzasadnienie techniczne: TODO

# Część 5 — Django models i migracje

Poniższe zadania wykonuj w projekcie Django. Notebook jest checklistą i miejscem na notatki. Nazwy modeli i pól zapisuj po angielsku. Komentarze możesz pisać po polsku.

## Zadanie 23 — Utworzenie aplikacji `orders`

- Biznes: wydzielasz moduł zamówień w projekcie.
- Cel techniczny: `python manage.py startapp orders`.
- Trudność: 1/5
- Szacowany czas: 8 min

Utwórz aplikację `orders` i dodaj ją do `INSTALLED_APPS`.

Wskazówki:
- Po utworzeniu aplikacji uruchom `python manage.py check`.
- Sprawdź, czy aplikacja jest wpisana w `settings.py`.
- Nie twórz jeszcze endpointów API.

In [ ]:
# Komendy do wykonania w terminalu, nie w notebooku:
# python manage.py startapp orders
# python manage.py check

# TODO: Zapisz tutaj, czy aplikacja została dodana do INSTALLED_APPS.
orders_app_added = None

## Zadanie 24 — Model `Customer`

- Biznes: zapisujesz klientów aplikacji.
- Cel techniczny: podstawowe pola modelu i `unique=True`.
- Trudność: 2/5
- Szacowany czas: 15 min

W pliku `orders/models.py` utwórz model `Customer` z polami: `email`, `full_name`, `city`, `state`.

Wskazówki:
- `email` powinien być unikalny.
- `city` i `state` mogą być opcjonalne na poziomie formularza.
- Dodaj metodę `__str__`.

In [ ]:
# Szkic do przeniesienia do orders/models.py

customer_model_draft = """
from django.db import models


class Customer(models.Model):
    # TODO: dodaj email jako EmailField z unique=True
    # TODO: dodaj full_name
    # TODO: dodaj city
    # TODO: dodaj state

    def __str__(self):
        # TODO: zwróć czytelną reprezentację klienta
        pass
"""

print(customer_model_draft)

## Zadanie 25 — Model `Product`

- Biznes: tworzysz katalog produktów.
- Cel techniczny: `DecimalField`, `TextChoices`, `BooleanField`.
- Trudność: 2/5
- Szacowany czas: 18 min

Utwórz model `Product` z polami: `name`, `category`, `price`, `is_active`.

Wskazówki:
- Do pieniędzy używaj `DecimalField`, nie `FloatField`.
- Kategorie zrób przez `models.TextChoices`.
- `is_active` powinno mieć wartość domyślną.

In [ ]:
product_model_draft = """
class Product(models.Model):
    class Category(models.TextChoices):
        # TODO: dodaj 3-4 kategorie, np. BOOKS, ELECTRONICS, HOME, OTHER
        pass

    # TODO: dodaj name
    # TODO: dodaj category z choices
    # TODO: dodaj price jako DecimalField
    # TODO: dodaj is_active

    def __str__(self):
        # TODO: zwróć nazwę produktu
        pass
"""

print(product_model_draft)

## Zadanie 26 — Model `Order` z `ForeignKey`

- Biznes: klient składa zamówienia.
- Cel techniczny: relacja 1:N i `choices` dla statusu.
- Trudność: 2/5
- Szacowany czas: 18 min

Utwórz model `Order`, który ma FK do `Customer`, status oraz datę utworzenia.

Wskazówki:
- Pole nazwij `customer`, nie `customer_id`.
- Rozważ `on_delete=models.PROTECT`.
- Dodaj `related_name='orders'`.

In [ ]:
order_model_draft = """
class Order(models.Model):
    class Status(models.TextChoices):
        # TODO: dodaj statusy DRAFT, PAID, SHIPPED, DELIVERED, CANCELED
        pass

    # TODO: dodaj ForeignKey do Customer
    # TODO: dodaj status z choices
    # TODO: dodaj created_at

    def __str__(self):
        # TODO: zwróć czytelny opis zamówienia
        pass
"""

print(order_model_draft)

## Zadanie 27 — Model `OrderItem` jako relacja pośrednia

- Biznes: zamówienie składa się z pozycji.
- Cel techniczny: dwa FK i pola relacji.
- Trudność: 3/5
- Szacowany czas: 25 min

Utwórz model `OrderItem`, który łączy `Order` i `Product`. Dodaj `quantity` oraz `unit_price`.

Wskazówki:
- FK do `Order` może mieć `on_delete=models.CASCADE`.
- FK do `Product` często lepiej zabezpieczyć przez `PROTECT`.
- `unit_price` zapisuje cenę z momentu zakupu, a nie aktualną cenę katalogową.

In [ ]:
order_item_model_draft = """
class OrderItem(models.Model):
    # TODO: dodaj ForeignKey do Order
    # TODO: dodaj ForeignKey do Product
    # TODO: dodaj quantity jako PositiveIntegerField
    # TODO: dodaj unit_price jako DecimalField

    def __str__(self):
        # TODO: zwróć czytelny opis pozycji zamówienia
        pass
"""

print(order_item_model_draft)

## Zadanie 28 — Constraints dla jakości danych

- Biznes: baza ma pilnować podstawowych reguł zamówienia.
- Cel techniczny: `CheckConstraint`, `UniqueConstraint`.
- Trudność: 4/5
- Szacowany czas: 25 min

Dodaj constraints do `OrderItem`:

- `quantity > 0`,
- `unit_price >= 0`,
- jeden produkt może pojawić się tylko raz w jednym zamówieniu.

Wskazówki:
- Użyj `from django.db.models import Q`.
- Constraints dodaje się w klasie `Meta`.
- Nazwy constraintów powinny być opisowe.

In [ ]:
constraints_draft = """
from django.db.models import Q


class OrderItem(models.Model):
    # TODO: pola modelu

    class Meta:
        constraints = [
            # TODO: CheckConstraint dla quantity > 0
            # TODO: CheckConstraint dla unit_price >= 0
            # TODO: UniqueConstraint dla order + product
        ]
"""

print(constraints_draft)

# Część 6 — QuerySets w Django ORM

## Zadanie 29 — Pierwsze QuerySets

- Biznes: pobierasz dane do przyszłego endpointu API.
- Cel techniczny: `all`, `filter`, `get`, `count`, `exists`.
- Trudność: 2/5
- Szacowany czas: 15 min

W Django shell wykonaj kilka podstawowych zapytań na modelach `Customer`, `Product`, `Order`.

Wskazówki:
- Uruchom `python manage.py shell`.
- Zaimportuj modele.
- Sprawdź różnicę między `filter` a `get`.

In [ ]:
queryset_draft_1 = """
from orders.models import Customer, Product, Order, OrderItem

# TODO: pobierz wszystkich klientów

# TODO: policz produkty

# TODO: znajdź zamówienia o wybranym statusie

# TODO: sprawdź, czy istnieje produkt aktywny

# TODO: użyj get dla jednego obiektu i zaobserwuj możliwe wyjątki
"""

print(queryset_draft_1)

## Zadanie 30 — Filtrowanie po relacji

- Biznes: support chce zobaczyć zamówienia konkretnego klienta.
- Cel techniczny: lookup przez `__`.
- Trudność: 3/5
- Szacowany czas: 18 min

Napisz QuerySet, który pobiera zamówienia klienta po adresie e-mail. Potem dodaj filtr po statusie.

Wskazówki:
- Użyj `customer__email`.
- To odpowiada JOIN-owi w SQL.
- Dodaj `.order_by('-created_at')`.

In [ ]:
queryset_draft_2 = """
# TODO: zamówienia klienta po emailu

# TODO: zamówienia klienta po emailu i statusie

# TODO: posortuj od najnowszych
"""

print(queryset_draft_2)

## Zadanie 31 — Sortowanie i ograniczanie wyników

- Biznes: panel admina pokazuje 10 ostatnich zamówień.
- Cel techniczny: `order_by`, slicing QuerySet.
- Trudność: 2/5
- Szacowany czas: 12 min

Napisz QuerySet zwracający 10 najnowszych zamówień.

Wskazówki:
- `-created_at` oznacza sortowanie malejące.
- Slicing QuerySetu przypomina listę Pythona.
- QuerySet jest leniwy, SQL wykona się dopiero przy ewaluacji.

In [ ]:
queryset_draft_3 = """
# TODO: 10 najnowszych zamówień

# TODO: wypisz ich id, status i klienta
"""

print(queryset_draft_3)

## Zadanie 32 — Agregacje w ORM

- Biznes: raport wartości zamówień i liczby pozycji.
- Cel techniczny: `values`, `annotate`, `Sum`, `Count`.
- Trudność: 4/5
- Szacowany czas: 30 min

Napisz QuerySet, który grupuje zamówienia po statusie i liczy liczbę zamówień w każdym statusie. Następnie przygotuj szkic raportu wartości pozycji zamówień.

Wskazówki:
- Zaimportuj `Count` i `Sum` z `django.db.models`.
- `values('status')` wyznacza grupowanie.
- `annotate(...)` dodaje wyliczenie.

In [ ]:
queryset_draft_4 = """
from django.db.models import Count, Sum

# TODO: policz zamówienia po statusie

# TODO: policz liczbę pozycji zamówień po produkcie lub statusie

# TODO: sprawdź SQL przez print(queryset.query)
"""

print(queryset_draft_4)

## Zadanie 33 — Problem N+1

- Biznes: endpoint listy zamówień wykonuje zbyt dużo zapytań.
- Cel techniczny: `select_related`, `prefetch_related`.
- Trudność: 4/5
- Szacowany czas: 25 min

Przygotuj dwa QuerySety:

1. Lista zamówień z klientem — użyj `select_related`.
2. Lista zamówień z pozycjami — użyj `prefetch_related`.

Wskazówki:
- FK do jednego obiektu: zwykle `select_related`.
- Reverse FK lub wiele obiektów: zwykle `prefetch_related`.
- W Django Debug Toolbar albo w logach można obserwować liczbę zapytań.

In [ ]:
queryset_draft_5 = """
# TODO: zamówienia z klientem przez select_related

# TODO: zamówienia z pozycjami przez prefetch_related

# TODO: opisz, dlaczego używasz różnych metod
"""

print(queryset_draft_5)

# Część 7 — Migracje Django

## Zadanie 34 — Pierwsza migracja

- Biznes: tworzysz strukturę bazy dla modułu zamówień.
- Cel techniczny: `makemigrations`, `migrate`.
- Trudność: 2/5
- Szacowany czas: 15 min

Po dodaniu modeli wygeneruj i uruchom migrację.

Wskazówki:
- Najpierw `python manage.py check`.
- Potem `python manage.py makemigrations`.
- Na końcu `python manage.py migrate`.

In [ ]:
migration_commands_1 = """
python manage.py check
python manage.py makemigrations
python manage.py migrate
"""

print(migration_commands_1)

# TODO: zapisz nazwę wygenerowanej migracji
first_migration_name = None

## Zadanie 35 — Analiza pliku migracji

- Biznes: rozumiesz, co Django chce zmienić w bazie.
- Cel techniczny: czytanie operacji migracji.
- Trudność: 3/5
- Szacowany czas: 15 min

Otwórz wygenerowany plik migracji w katalogu `orders/migrations/` i wypisz, jakie operacje zawiera.

Wskazówki:
- Szukaj `CreateModel`.
- Sprawdź, jakie pola stworzy Django.
- Sprawdź, czy constraints są widoczne w migracji.

### Twoje notatki z migracji

- Nazwa pliku migracji: TODO
- Operacje:
  - TODO
  - TODO
  - TODO
- Czy constraints są w migracji? TODO

## Zadanie 36 — Zmiana modelu i kolejna migracja

- Biznes: produkt dostaje nowe pole `is_featured` albo `sku`.
- Cel techniczny: migracja zmiany schematu.
- Trudność: 3/5
- Szacowany czas: 20 min

Dodaj nowe pole do modelu `Product`, wygeneruj migrację i zastosuj ją.

Wskazówki:
- Jeśli pole jest wymagane, potrzebny może być `default`.
- Przeczytaj komunikat Django, jeśli zapyta o wartość domyślną.
- Po migracji uruchom `python manage.py check`.

In [ ]:
migration_commands_2 = """
# TODO: dodaj pole w models.py, np.:
# is_featured = models.BooleanField(default=False)

python manage.py makemigrations
python manage.py migrate
python manage.py check
"""

print(migration_commands_2)

# TODO: zapisz nazwę drugiej migracji
second_migration_name = None

## Zadanie 37 — `sqlmigrate`, czyli SQL generowany przez Django

- Biznes: backend developer rozumie, co ORM robi pod spodem.
- Cel techniczny: `python manage.py sqlmigrate`.
- Trudność: 3/5
- Szacowany czas: 15 min

Użyj `sqlmigrate`, żeby zobaczyć SQL wygenerowany dla jednej migracji aplikacji `orders`.

Wskazówki:
- Potrzebujesz nazwy aplikacji i numeru migracji.
- Przykład: `python manage.py sqlmigrate orders 0001`.
- Nie musisz rozumieć każdego szczegółu, ale wskaż `CREATE TABLE`, FK i constraints.

In [ ]:
sqlmigrate_command = """
python manage.py sqlmigrate orders 0001
"""

print(sqlmigrate_command)

# TODO: Wklej w notatkach fragmenty, które rozpoznajesz jako CREATE TABLE, FK i constraints.

### Notatki z `sqlmigrate`

- Jakie tabele powstaną? TODO
- Gdzie widzisz klucze obce? TODO
- Gdzie widzisz constraints? TODO
- Co było zaskakujące? TODO

# Część 8 — Mini-projekt do projektu końcowego

## Zadanie 38 — Model danych do projektu końcowego

- Biznes: Twój projekt końcowy potrzebuje trwałego modelu danych.
- Cel techniczny: ERD + Django models + pierwsza migracja.
- Trudność: 4/5
- Szacowany czas: 45 min

Zaprojektuj i zaimplementuj minimalny model danych dla swojego projektu końcowego.

Wymagania minimalne:

- minimum 3 modele,
- minimum 2 relacje,
- minimum 1 `ForeignKey`,
- minimum 1 constraint,
- poprawnie wygenerowana migracja,
- poprawnie uruchomiona migracja,
- minimum 5 QuerySetów zapisanych jako notatki lub kod.

Wskazówki:
- Zacznij od rzeczowników w opisie projektu.
- Nie projektuj od razu całego systemu.
- Zadbaj o nazwy modeli i relacji.

### Szablon dla projektu końcowego

#### Nazwa projektu

TODO

#### Modele

| Model | Odpowiedzialność biznesowa | Najważniejsze pola |
|---|---|---|
| TODO | TODO | TODO |
| TODO | TODO | TODO |
| TODO | TODO | TODO |

#### Relacje

| Relacja | Typ | Dlaczego tak? |
|---|---|---|
| TODO | 1:N / N:M | TODO |
| TODO | 1:N / N:M | TODO |

#### Constraints

| Constraint | Model | Reguła biznesowa |
|---|---|---|
| TODO | TODO | TODO |

#### QuerySety potrzebne do API

1. TODO
2. TODO
3. TODO
4. TODO
5. TODO

# Zadania dodatkowe dla chętnych

## Bonus 1 — Porównanie SQL i ORM

- Biznes: sprawdzasz, czy ORM generuje sensowne zapytanie.
- Cel techniczny: `print(queryset.query)`.
- Trudność: 3/5
- Szacowany czas: 15 min

Wybierz jeden QuerySet z zadania 30, 31 albo 32. Wypisz SQL generowany przez Django i porównaj go z zapytaniem SQL z soboty.

TODO:
- QuerySet: ...
- Wygenerowany SQL: ...
- Co jest podobne do ręcznego SQL-a? ...
- Co Django zrobiło za Ciebie? ...

## Bonus 2 — AI jako reviewer modelu

- Biznes: używasz AI do szybszego review, ale samodzielnie weryfikujesz wynik.
- Cel techniczny: analiza modelu Django i migracji.
- Trudność: 3/5
- Szacowany czas: 15 min

Wklej swój model Django do wybranego narzędzia AI i poproś o review.

Przykładowy prompt:

Przejrzyj ten model Django. Wskaż potencjalne problemy z relacjami, nazwami pól, constraints, null/blank i migracjami. Odpowiedz jak mentor backendu dla osoby początkującej.

Po otrzymaniu odpowiedzi zapisz:

- Która sugestia była trafna? TODO
- Która sugestia była ryzykowna albo niepotrzebna? TODO
- Co faktycznie zmieniasz w kodzie? TODO